## Raw data sets treatment to create useable new spectral data bases

In [24]:
# Force the working directory to be the one of the Github repo
import os
os.chdir("/home/robinr/Desktop/VSCode/CIRAD_PhD_Robin")
print("Working dir:", os.getcwd())

import csv
from pathlib import Path
import pandas as pd
from Scripts_python.Splitting.kennard_stone import kennard_stone

# import warnings filter
from warnings import simplefilter
# ignore all future warnings
simplefilter(action='ignore', category=FutureWarning)
simplefilter(action='ignore', category=UserWarning)
simplefilter(action="ignore", category=RuntimeWarning)

Working dir: /home/robinr/Desktop/VSCode/CIRAD_PhD_Robin


In [3]:
# Function to load a CSV file with automatic separator detection

def load_csv_auto_sep(mode, data_source, type_data, verbose=True, delimiter=None, index_col=None):

    ## Importation of the datasets with the adapted path
    file_name = Path("Data/%s/%s"% (mode,data_source))
    full_path = str(file_name.resolve()).replace("\\", "/")
    path = full_path + "/%s.csv" % type_data
    
    with open(path, 'r', newline='', encoding='utf-8-sig') as f:

        if delimiter is not None:
            sep = delimiter
        
        else:
            # Read a small portion of the file to detect the separator
            excerpt = f.read(1024)
            f.seek(0)  # return to the beginning of the file

            # Detection of the dialect
            dialect = csv.Sniffer().sniff(excerpt)
            sep = dialect.delimiter

        if verbose: print("Detected separator for %s: %s" % (type_data, sep))
        
        # Load the file with pandas
        df = pd.read_csv(f, delimiter=sep, index_col=index_col)

        if type_data[0]=='Y' and len(df.columns) > 1:
            # Drop the useless column if it exists
            df = df.drop(columns=[df.columns[1]])
        
        return df

## Grapevines

#### Innospectra measurements

In [70]:
# Load Innospectra measurements
df_nirs = load_csv_auto_sep(mode="Raw", data_source="Grapevines_chloride", type_data="innospectra_reflectance", verbose=True, delimiter=None, index_col=0)

# Load the chloridometer readings
df_chloride = load_csv_auto_sep(mode="Raw", data_source="Grapevines_chloride", type_data="chloridometer_readings", verbose=True, delimiter=None)

# Drop missing measures with missing values
to_drop = df_nirs[df_nirs["pot number"] == 266].index
df_nirs.drop(labels=to_drop, inplace=True)
to_drop = df_chloride[df_chloride["pot number"] == 266].index
df_chloride.drop(labels=to_drop, inplace=True)

# Keep spectra only
df_nirs.drop(labels="pot number", axis=1, inplace=True)

# Keep the averaged chloride content measure only
df_chloride = df_chloride["average"]

Detected separator for innospectra_reflectance: ,
Detected separator for chloridometer_readings: ,


In [ ]:
# Number of calibration samples (70% of dataset)
n_total = df_nirs.shape[0]
n_cal = int(0.7 * n_total)

# Apply Kennard-Stone selection on X
cal_indices = kennard_stone(df_nirs.values, n_cal)

# Validation set = the rest
val_indices = list(set(range(n_total)) - set(cal_indices))

# Split X and Y into calibration and validation sets
Xcal = df_nirs.iloc[cal_indices, :]
Ycal = df_chloride.iloc[cal_indices]

Xval = df_nirs.iloc[val_indices, :]
Yval = df_chloride.iloc[val_indices]

# Save the four CSV files
path = os.path.join("Data", "Regression", "grapevine_chloride_260_KS")
os.makedirs(path, exist_ok=True)
Xcal.to_csv(os.path.join(path, "Xcal.csv"), index=False)
Ycal.to_csv(os.path.join(path, "Ycal.csv"), index=False)
Xval.to_csv(os.path.join(path, "Xval.csv"), index=False)
Yval.to_csv(os.path.join(path, "Yval.csv"), index=False)

print("Files Xcal.csv, Ycal.csv, Xval.csv, Yval.csv have been generated for the Innospectra measures.")

Files Xcal.csv, Ycal.csv, Xval.csv, Yval.csv have been generated for the Innospectra measures.
1487.5826976017127


#### SVC measurements

In [ ]:
# Load SVC measurements
df_nirs_1 = load_csv_auto_sep(mode="Raw", data_source="Grapevines_chloride", type_data="230606_svc_reflectance", verbose=True, delimiter=None)
df_nirs_2 = load_csv_auto_sep(mode="Raw", data_source="Grapevines_chloride", type_data="230718_svc_reflectance", verbose=True, delimiter=None)

# Load the chloridometer readings
df_chloride_1 = load_csv_auto_sep(mode="Raw", data_source="Grapevines_chloride", type_data="chloridometer_readings", verbose=True, delimiter=None)
df_chloride_1.rename(columns={"svc_id": "scan"}, inplace=True)

df_chloride_2 = load_csv_auto_sep(mode="Raw", data_source="Grapevines_chloride", type_data="chloridometer_readings (1)", verbose=True, delimiter=None)
df_chloride_2.rename(columns={"svc_id": "scan"}, inplace=True)

df1 = df_chloride_1.merge(df_nirs_1, how="outer", on="scan")
df1.dropna(how="any", inplace=True)

df2 = df_chloride_2.merge(df_nirs_2, how="outer", on="scan")
df2.dropna(how="any", inplace=True)

df = pd.concat([df1, df2], axis=0)

X = df.iloc[:,9:]
Y = df["average"]

# Number of calibration samples (70% of dataset)
n_total = X.shape[0]
n_cal = int(0.7 * n_total)

# Apply Kennard-Stone selection on X
cal_indices = kennard_stone(X.values, n_cal)

# Validation set = the rest
val_indices = list(set(range(n_total)) - set(cal_indices))

# Split X and Y into calibration and validation sets
Xcal = X.iloc[cal_indices, :]
Ycal = Y.iloc[cal_indices]

Xval = X.iloc[val_indices, :]
Yval = Y.iloc[val_indices]

# Save the four CSV files
path = os.path.join("Data", "Regression", "grapevine_chloride_556_KS")
os.makedirs(path, exist_ok=True)
Xcal.to_csv(os.path.join(path, "Xcal.csv"), index=False)
Ycal.to_csv(os.path.join(path, "Ycal.csv"), index=False)
Xval.to_csv(os.path.join(path, "Xval.csv"), index=False)
Yval.to_csv(os.path.join(path, "Yval.csv"), index=False)

print("Files Xcal.csv, Ycal.csv, Xval.csv, Yval.csv have been generated for the SVC measures.")

Detected separator for 230606_svc_reflectance: ,
Detected separator for 230718_svc_reflectance: ,
Detected separator for chloridometer_readings: ,
Detected separator for chloridometer_readings (1): ,
Files Xcal.csv, Ycal.csv, Xval.csv, Yval.csv have been generated for the SVC measures.


np.float64(1720.2002724193042)

## Milk

In [93]:
display(df)

,Cow_ID,Fat,Prot,Lact,SCC,Urea,Milk_yield,Milk_Interv,SET,Time_Dark,...,Trans_White_247,Trans_White_248,Trans_White_249,Trans_White_250,Trans_White_251,Trans_White_252,Trans_White_253,Trans_White_254,Trans_White_255,Trans_White_256
0,57017,2.79,3.55,4.84,87,30,12.73,27256,1,1495647915,...,22039,21978,21916,21872,21818,21780,21758,21720,21698,21672
1,53330,4.70,3.29,4.94,14,29,15.00,34359,1,1495648408,...,22039,21978,21915,21872,21817,21779,21757,21719,21697,21672
2,59129,3.35,3.21,4.96,21,25,13.26,33081,1,1495648930,...,22039,21978,21916,21872,21817,21780,21758,21720,21698,21672
3,53333,2.69,3.02,4.83,9,31,18.19,30596,1,1495649484,...,22040,21979,21916,21872,21818,21780,21758,21720,21698,21672
4,57013,4.60,3.83,4.70,14,28,13.22,39668,1,1495651684,...,22039,21978,21915,21871,21816,21779,21757,21719,21696,21671
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1219,57019,3.59,3.70,4.57,18,22,12.63,31665,8,1501141930,...,22035,21974,21910,21866,21812,21774,21752,21714,21691,21667
1220,59131,4.02,3.62,4.81,11,21,9.99,36471,8,1501142498,...,22033,21972,21909,21866,21811,21774,21751,21713,21691,21665
1221,59130,3.40,2.94,4.63,14,26,14.04,21407,8,1501143001,...,22033,21972,21909,21865,21811,21773,21751,21713,21690,21665
1222,53337,3.44,3.18,4.70,12,24,15.04,29863,8,1501143442,...,22034,21973,21910,21866,21812,21774,21752,21714,21691,21666


In [92]:
### Load the raw dataset
df = load_csv_auto_sep(mode="Raw", data_source="milk", type_data="data_table", verbose=True, delimiter=None)

### CREATE DATASETS FOR FAT CONTENT ###
# Construct the target vector
Y = df["Fat"]

# Construct the spectra dataset with appropriate column names
X = df.loc[:,"Trans_Tot_1":"Trans_Tot_256"]
X.rename(columns={f"Trans_Tot_{i}": f"X_{round(960 + 2.86*i, 1)}" for i in range(256)}, inplace=True)

# Split the dataset with the Kennard Stone method
# Number of calibration samples (70% of dataset)
n_total = X.shape[0]
n_cal = int(0.7 * n_total)

# Apply Kennard-Stone selection on X
cal_indices = kennard_stone(X.values, n_cal)

# Validation set = the rest
val_indices = list(set(range(n_total)) - set(cal_indices))

# Split X and Y into calibration and validation sets
Xcal = X.iloc[cal_indices, :]
Ycal = Y.iloc[cal_indices]

Xval = X.iloc[val_indices, :]
Yval = Y.iloc[val_indices]

# Save the four CSV files
path = os.path.join("Data", "Regression", "Milk_Fat_1224_KS")
os.makedirs(path, exist_ok=True)
Xcal.to_csv(os.path.join(path, "Xcal.csv"), index=False)
Ycal.to_csv(os.path.join(path, "Ycal.csv"), index=False)
Xval.to_csv(os.path.join(path, "Xval.csv"), index=False)
Yval.to_csv(os.path.join(path, "Yval.csv"), index=False)

print("Files Xcal.csv, Ycal.csv, Xval.csv, Yval.csv have been generated for the Fat content.")

Detected separator for data_table: ,
Files Xcal.csv, Ycal.csv, Xval.csv, Yval.csv have been generated for the Fat content.


In [ ]:
### CREATE DATASETS FOR the targeted analyte ###
name_target = "SCC"

# Construct the target vector
Y = df[name_target]

# Construct the spectra dataset with appropriate column names
X = df.loc[:,"Trans_Tot_1":"Trans_Tot_256"]
X.rename(columns={f"Trans_Tot_{i}": f"X_{round(960 + 2.86*i, 1)}" for i in range(256)}, inplace=True)

# Split the dataset with the Kennard Stone method
# Number of calibration samples (70% of dataset)
n_total = X.shape[0]
n_cal = int(0.7 * n_total)

# Apply Kennard-Stone selection on X
cal_indices = kennard_stone(X.values, n_cal)

# Validation set = the rest
val_indices = list(set(range(n_total)) - set(cal_indices))

# Split X and Y into calibration and validation sets
Xcal = X.iloc[cal_indices, :]
Ycal = Y.iloc[cal_indices]

Xval = X.iloc[val_indices, :]
Yval = Y.iloc[val_indices]

# Save the four CSV files
path = os.path.join("Data", "Regression", f"Milk_{name_target}_1224_KS")
os.makedirs(path, exist_ok=True)
Xcal.to_csv(os.path.join(path, "Xcal.csv"), index=False)
Ycal.to_csv(os.path.join(path, "Ycal.csv"), index=False)
Xval.to_csv(os.path.join(path, "Xval.csv"), index=False)
Yval.to_csv(os.path.join(path, "Yval.csv"), index=False)

print(f"Files Xcal.csv, Ycal.csv, Xval.csv, Yval.csv have been generated for the {name_target}.")

Files Xcal.csv, Ycal.csv, Xval.csv, Yval.csv have been generated for the Somatic Cell Count.


## Manure

In [ ]:
### CREATE DATASETS FOR the targeted analyte ###
name_target = "CaO"
type_animal = "Poultry" # "Poultry" or "Cattle"

# Read the xlsx file of chemical measurements
df_chem = pd.read_excel("Data/Raw/manure/chemical_analysis.xlsx")
df_chem = df_chem[df_chem["Manure_type"]==type_animal+" manure"]
df_chem = df_chem[["Id_sample", name_target]]

# Read the xlsx file of dry manure
df_nirs = pd.read_csv("Data/Raw/manure/spectra-1.csv", decimal=",", quotechar='"')

df = pd.merge(left=df_chem, right=df_nirs, how="inner", on="Id_sample")
df.drop("Id_sample", axis=1, inplace=True)

X = df.iloc[:,1:]
Y = df[name_target]

# Split the dataset with the Kennard Stone method
# Number of calibration samples (70% of dataset)
n_total = X.shape[0]
n_cal = int(0.7 * n_total)

# Apply Kennard-Stone selection on X
cal_indices = kennard_stone(X.values, n_cal)

# Validation set = the rest
val_indices = list(set(range(n_total)) - set(cal_indices))

# Split X and Y into calibration and validation sets
Xcal = X.iloc[cal_indices, :]
Ycal = Y.iloc[cal_indices]

Xval = X.iloc[val_indices, :]
Yval = Y.iloc[val_indices]

display(Xcal)

# Save the four CSV files
path = os.path.join("Data", "Regression", f"{type_animal}_manure_{name_target}_KS")
os.makedirs(path, exist_ok=True)
Xcal.to_csv(os.path.join(path, "Xcal.csv"), index=False)
Ycal.to_csv(os.path.join(path, "Ycal.csv"), index=False)
Xval.to_csv(os.path.join(path, "Xval.csv"), index=False)
Yval.to_csv(os.path.join(path, "Yval.csv"), index=False)

print(f"Files Xcal.csv, Ycal.csv, Xval.csv, Yval.csv have been generated for the {type_animal} manure targeting {name_target}.")

print("min : ", Y.min())
print("max : ", Y.max())
print("mean : ", Y.mean())

,852.78_nm,853.34_nm,853.9_nm,854.47_nm,855.03_nm,855.6_nm,856.16_nm,856.73_nm,857.29_nm,857.86_nm,...,2459.63_nm,2464.31_nm,2469_nm,2473.72_nm,2478.45_nm,2483.19_nm,2487.96_nm,2492.74_nm,2497.55_nm,2502.37_nm
43,0.378000,0.378470,0.379110,0.377722,0.376468,0.377235,0.378048,0.377751,0.376035,0.374485,...,0.678483,0.685738,0.692545,0.698833,0.704499,0.709389,0.713265,0.716068,0.717891,0.718802
133,1.368148,1.372899,1.371021,1.373636,1.381014,1.376762,1.369679,1.364258,1.362138,1.362441,...,1.289174,1.291623,1.294136,1.296400,1.298175,1.299467,1.300136,1.300180,1.299937,1.299432
7,0.695642,0.696157,0.694883,0.693826,0.692188,0.688642,0.685958,0.685706,0.685778,0.685548,...,1.161366,1.165635,1.169503,1.173053,1.176408,1.179114,1.180815,1.181706,1.182040,1.182028
93,0.898901,0.899481,0.901510,0.899416,0.894324,0.893282,0.895133,0.895633,0.895813,0.893246,...,1.371484,1.373766,1.375898,1.377668,1.379237,1.380319,1.380534,1.380235,1.379781,1.379025
101,0.637203,0.636882,0.637828,0.636840,0.633926,0.635122,0.634281,0.629176,0.628930,0.630023,...,0.869596,0.876756,0.883679,0.890120,0.895958,0.901084,0.905165,0.908044,0.909925,0.911005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31,0.668493,0.669296,0.672509,0.672841,0.670419,0.667944,0.667171,0.668013,0.668498,0.668361,...,1.019065,1.026172,1.032928,1.039253,1.045081,1.050120,1.054037,1.056764,1.058533,1.059594
135,1.023617,1.020799,1.018116,1.017558,1.016479,1.016087,1.015888,1.013405,1.009508,1.007330,...,1.322917,1.325319,1.327671,1.329796,1.331708,1.333151,1.333801,1.333999,1.333950,1.333510
88,0.720790,0.721266,0.721039,0.719494,0.719159,0.721351,0.721110,0.716976,0.716175,0.717451,...,1.087891,1.094337,1.100490,1.106318,1.111736,1.116331,1.119792,1.122316,1.124072,1.125047
79,0.821103,0.820955,0.817964,0.819315,0.822265,0.817766,0.812560,0.809616,0.808523,0.810303,...,1.054725,1.060586,1.066178,1.071316,1.075942,1.079892,1.083132,1.085661,1.087405,1.088392


Files Xcal.csv, Ycal.csv, Xval.csv, Yval.csv have been generated for the Poultry manure targeting CaO.
min :  3.66
max :  109.58
mean :  22.918819444444445


## Beef

In [61]:
import os
import pandas as pd
import numpy as np

# Read the xlsx file of chemical measurements
df = pd.read_excel("Data/Raw/Beef/Data NIR Marbling.xlsx")

# Remove the column "Animal Number"
df.drop("Animal Number", axis=1, inplace=True)

# Set new column names: column 0 keeps its original name, others take the value from row 1
df.columns = [df.columns[0]] + df.iloc[0, 1:].tolist()

# Remove the first row which contained new header names
df.drop(index=0, inplace=True)

# Reset index so iloc works correctly
df = df.reset_index(drop=True)

# Split X (features) and Y (target)
X = df.iloc[:, 1:]
Y = df.iloc[:, 0]

# Number of calibration samples (2/3 of dataset)
n_total = X.shape[0]
n_cal = int(2/3 * n_total)

# Ensure reproducibility for random splitting
np.random.seed(42)

# ---------------------------------------------------------
# Random split ensuring the max-Y sample is in calibration
# ---------------------------------------------------------

# *** Position *** of the maximum value of Y  
idx_max_y = int(np.argmax(Y.values))   # gives a position, safe for iloc

# All row positions
all_positions = np.arange(n_total)

# Positions excluding the max-Y sample
remaining_positions = np.setdiff1d(all_positions, [idx_max_y])

# Number of additional calibration samples needed
n_cal_remaining = n_cal - 1

# Randomly select the remaining calibration positions
cal_random = np.random.choice(remaining_positions, size=n_cal_remaining, replace=False)

# Final calibration positions
cal_positions = np.concatenate(([idx_max_y], cal_random))

# Validation positions
val_positions = np.setdiff1d(all_positions, cal_positions)

# Create calibration and validation sets
Xcal = X.iloc[cal_positions, :]
Ycal = Y.iloc[cal_positions]

Xval = X.iloc[val_positions, :]
Yval = Y.iloc[val_positions]

# Save the four CSV files
path = os.path.join("Data", "Regression", "Beef_Marbling_RandomSplit")
os.makedirs(path, exist_ok=True)

Xcal.to_csv(os.path.join(path, "Xcal.csv"), index=False)
Ycal.to_csv(os.path.join(path, "Ycal.csv"), index=False)
Xval.to_csv(os.path.join(path, "Xval.csv"), index=False)
Yval.to_csv(os.path.join(path, "Yval.csv"), index=False)

print("Files Xcal.csv, Ycal.csv, Xval.csv, Yval.csv have been generated for the beef marbling dataset.")
print("min : ", Y.min())
print("max : ", Y.max())
print("mean : ", Y.mean())


Files Xcal.csv, Ycal.csv, Xval.csv, Yval.csv have been generated for the beef marbling dataset.
min :  100
max :  810
mean :  334.640625


## else

In [ ]:
from scipy.io import loadmat
from scipy.io import loadmat
import numpy as np

# Chargement du fichier
data = loadmat('Data/Raw/mat/CGL_nir.mat')

# Récupérer l'objet 'Spectra'
spectra_struct = data['Spectra']

# Afficher les noms de champs disponibles
print("Champs disponibles dans 'Spectra' :", dir(spectra_struct))

# Hypothèse : les champs s'appellent souvent 'data', 'wavelength' ou similaire
# Affichons quelques détails
try:
    spectra_array = spectra_struct.data  # ou spectra_struct.y si nécessaire
    wavelengths = spectra_struct.x  # ou .wavelength, selon le nom exact

    print("Spectra shape:", spectra_array.shape)
    print("Wavelengths shape:", wavelengths.shape)

    # Stack sous forme (échantillons, longueurs d’onde)
    # selon orientation : (wavelengths,) x (spectra,) ou l'inverse
    if spectra_array.shape[0] == wavelengths.shape[0]:
        spectra_np = np.array(spectra_array)
    else:
        spectra_np = np.array(spectra_array).T  # transposé si nécessaire

    print("Final stacked array shape (spectra x wavelengths):", spectra_np.shape)

except AttributeError as e:
    print("Impossible d'accéder aux champs : ", e)

Champs disponibles dans 'Spectra' : ['T', '__abs__', '__add__', '__and__', '__array__', '__array_finalize__', '__array_function__', '__array_interface__', '__array_namespace__', '__array_priority__', '__array_struct__', '__array_ufunc__', '__array_wrap__', '__bool__', '__buffer__', '__class__', '__class_getitem__', '__complex__', '__contains__', '__copy__', '__deepcopy__', '__delattr__', '__delitem__', '__dict__', '__dir__', '__divmod__', '__dlpack__', '__dlpack_device__', '__doc__', '__eq__', '__float__', '__floordiv__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__iadd__', '__iand__', '__ifloordiv__', '__ilshift__', '__imatmul__', '__imod__', '__imul__', '__index__', '__init__', '__init_subclass__', '__int__', '__invert__', '__ior__', '__ipow__', '__irshift__', '__isub__', '__iter__', '__itruediv__', '__ixor__', '__le__', '__len__', '__lshift__', '__lt__', '__matmul__', '__mod__', '__module__', '__mul__', '__ne__', '__neg__', '__